# Corpus-Size Scalability

Graph stage re-run on random samples of 500, 1,000, 5,000, 10,000 and 25,000 documents (ten per size, seeds 42 to 51) and on the full corpus, with tau = 0.40, w >= 20 and the Louvain setting of the main analysis. Each run records the structural quantities of graph and backbone, runtime and peak memory (Section IV-D2, Table 8; Appendix A-D).

Input: `EID_KEYWORDS.xlsx`. Helpers in `scripts/e3_scalability.py`. Outputs in `results/e3_scalability/`. Gate: the full corpus must reproduce the published CRS exactly before any sample runs.

The manuscript quotes the timings of the run made on an 8-core Apple silicon laptop, kept in `results/e3_scalability/paper_run_macos/`; the last cells compare this run with it. Structural quantities must agree exactly; runtimes and memory depend on the machine. The runtime-isolation helper of the original script (`scripts/e3_runtime.py`) is not activated here because two pinned packages differ in patch version; the versions used are in `metadata.json`.

In [1]:
# ============================================================
# CONFIGURATION AND IMPORTS
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "4"          # four notebooks run concurrently on this machine
os.environ["HF_HUB_OFFLINE"] = "1"           # the embedding model must already be cached
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MPLCONFIGDIR"] = "results/e3_scalability/.matplotlib"   # gitignored

import sys
sys.path.insert(0, "scripts")

import importlib.metadata
import json
import platform
import random
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

import e3_scalability as e3      # corpus, sampling, metrics, gate, summaries, figures
import crs_reference as ref      # verbatim graph functions of notebooks 2, 3 and 5

OUT = Path("results/e3_scalability")
ARCHIVE = OUT / "paper_run_macos"         # run reported in the manuscript; never written here
assert OUT.resolve() == e3.OUT.resolve()

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print("Model:           ", e3.MODEL, "@", e3.REVISION)
print("tau:             ", e3.TAU)
print("Backbone w >=:   ", e3.BACKBONE)
print("Louvain seed:    ", e3.LOUVAIN_SEED)
print("Sample sizes:    ", e3.SIZES, "+ full corpus")
print("Replicate seeds: ", e3.SEEDS)

/home/mat/academic-writing/papers/from_text_to_structure/data_repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model:            sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 @ e8f8c211226b894fcb81acc59f3b34ba3efd5f42
tau:              0.4
Backbone w >=:    20
Louvain seed:     42
Sample sizes:     [500, 1000, 5000, 10000, 25000] + full corpus
Replicate seeds:  [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


In [2]:
# ============================================================
# PROTECTED INPUTS, REFERENCE FUNCTIONS AND ANALYSED CORPUS
# ============================================================
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
hashes = e3.protected()
source_hashes = {p: e3.digest(p) for p in
                 ["scripts/e3_scalability.py", "scripts/crs_reference.py", "scripts/e3_runtime.py"]}
reference_checks = e3.reference_audit()     # AST of crs_reference == notebooks 2, 3, 5
print("Reference functions AST-identical to the original notebooks:")
print(pd.DataFrame(reference_checks).T.to_string())

before_setup = time.perf_counter()
print("\nReading the original keyword file...")
corpus, corpus_audit = e3.load_corpus()
print(f"Raw Excel rows = {len(corpus_audit)} | analysed corpus N = {len(corpus)} | "
      f"excluded (empty parsed list) = {int((~corpus_audit.included_in_reference_corpus).sum())}")
assert all(n < len(corpus) for n in e3.SIZES)

OUT.mkdir(parents=True, exist_ok=True)
(OUT / "samples").mkdir(exist_ok=True)
corpus_audit.to_csv(OUT / "corpus_audit.csv", index=False)

# The protected inputs must be the ones the archived (manuscript) run used.
archived_metadata = json.loads((ARCHIVE / "metadata.json").read_text())
same_inputs = pd.Series({k: hashes.get(k) == v for k, v in archived_metadata["protected_sha256"].items()},
                        name="sha256 identical to the archived run")
print("\n" + same_inputs.to_string())
assert same_inputs.all(), "protected inputs differ from the archived run"

Reference functions AST-identical to the original notebooks:
                                   notebook ast_identical
parse_keywords                 2. CRS.ipynb          True
build_backbone        3. w_THRESHOLDS.ipynb          True
build_crs_for_tau  5. Tau_SENSITIVITY.ipynb          True

Reading the original keyword file...


Raw Excel rows = 52947 | analysed corpus N = 52946 | excluded (empty parsed list) = 1

1. LLMS.ipynb                   True
2. CRS.ipynb                    True
3. w_THRESHOLDS.ipynb           True
4. METRICS.ipynb                True
5. Tau_SENSITIVITY.ipynb        True
6. GROUND_TRUTH_INSPEC.ipynb    True
7. HUMAN_EVAL_M1.ipynb          True
EID_KEYWORDS.xlsx               True
dataset_inspec.csv              True
human_eval_M1_8b.csv            True
inspec_llama-3.1-8b-EN.csv      True


In [3]:
# ============================================================
# CONFIGURATION AND METADATA RECORDS, EMBEDDING MODEL
# ============================================================
versions = {name: importlib.metadata.version(name) for name in
            ["numpy", "pandas", "scipy", "scikit-learn", "networkx", "python-louvain", "torch",
             "sentence-transformers", "transformers", "huggingface-hub", "openpyxl", "matplotlib", "psutil"]}

# Pinned dependency closure of the macOS run vs the packages installed here (informative only;
# the isolation of e3_runtime is not activated, see the introduction).
pinned = dict(line.split("==") for line in (OUT / "environment_repair_requirements.txt").read_text().splitlines()
              if line and not line.startswith("#"))
def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
pin_mismatch = {name: {"pinned": v, "installed": installed(name)} for name, v in pinned.items() if installed(name) != v}
print("Pinned packages of the macOS closure whose installed version differs here:")
print(json.dumps(pin_mismatch, indent=2))

configuration = {"model": e3.MODEL, "model_revision": e3.REVISION, "tau": e3.TAU, "backbone_threshold": e3.BACKBONE,
                 "louvain_seed": e3.LOUVAIN_SEED, "louvain_weight": "weight", "louvain_resolution": 1.0,
                 "sizes": e3.SIZES + [len(corpus)], "replicate_seeds": e3.SEEDS, "full_corpus_runs": 1,
                 "sampling": "default_rng(SeedSequence([seed,n])).choice(N,n,replace=False); sort original row order",
                 "sampling_nested": False, "embedding_batch_size": 32, "normalize_embeddings": True,
                 "embedding_reuse_between_runs": False, "embedding_strategy": "unique sorted vocabulary once per run, as notebook 5",
                 "device": "cpu", "torch_threads": 4, "random_seed": 42,
                 "modularity_and_n_communities_scope": "backbone, exactly as notebooks 4/5",
                 "mean_degree_and_mean_weighted_degree_scope": "full graph; same degree formulas, additional scope",
                 "clustering_coefficient": "additional descriptive nx.average_clustering(full_graph,weight=None,count_zeros=True); not in original notebooks",
                 "empty_backbone": "retain zero counts/fraction; modularity and mean degrees undefined (NaN), no lowered threshold",
                 "edgeless_lcc_convention": "0 nodes/edges/fraction, as notebook 5",
                 "runtime_scope": "per-run vocabulary, embedding, graph aggregation, backbone and metrics; shared loading and sample CSV I/O excluded",
                 "sd_ddof": 1, "singleton_sd": None, "peak_memory": "RSS sampled every 50ms; includes shared loaded model; not isolated allocation peak"}
assert configuration == json.loads((ARCHIVE / "configuration.json").read_text()), "configuration differs from the archived run"

metadata = {"started_utc": datetime.now(timezone.utc).isoformat(), "branch": branch,
            "python": platform.python_version(), "platform": platform.platform(), "cpu_count": os.cpu_count(),
            "packages": versions, "raw_excel_rows": len(corpus_audit), "analyzed_corpus_rows": len(corpus),
            "excluded_empty_parsed_lists": int((~corpus_audit.included_in_reference_corpus).sum()),
            "protected_sha256": hashes, "source_sha256": source_hashes,
            "runtime_isolation": {"strategy": "not activated: shared .venv imported directly (e3_runtime symlink closure was a macOS repair)",
                                  "directory": None, "linked_top_level_entries": 0,
                                  "pinned_closure_mismatches": pin_mismatch},
            "notebook": "13. CORPUS_SIZE_SCALABILITY.ipynb",
            "llm_calls": 0, "paid_api_calls": 0, "model_downloads": 0, "status": "running"}
e3.json_write(OUT / "configuration.json", configuration)
e3.json_write(OUT / "metadata.json", metadata)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
try:
    model = SentenceTransformer(e3.MODEL, revision=e3.REVISION, device="cpu", local_files_only=True)
except Exception as exc:
    raise RuntimeError(f"Local model {e3.MODEL}@{e3.REVISION} not found in the Hugging Face cache; "
                       f"it is not downloaded automatically: {exc}") from exc
metadata["shared_setup_seconds"] = time.perf_counter() - before_setup
metadata["model_max_seq_length"] = model.max_seq_length
e3.json_write(OUT / "metadata.json", metadata)
print(f"\nModel loaded; shared setup {metadata['shared_setup_seconds']:.1f} s; "
      f"platform {metadata['platform']}; cpu_count {metadata['cpu_count']}; torch threads {torch.get_num_threads()}")

Pinned packages of the macOS closure whose installed version differs here:
{
  "filelock": {
    "pinned": "3.32.5",
    "installed": "3.32.3"
  },
  "narwhals": {
    "pinned": "2.25.0",
    "installed": "2.26.0"
  },
  "torch": {
    "pinned": "2.8.0",
    "installed": "2.8.0+cpu"
  },
  "setuptools": {
    "pinned": "84.0.0",
    "installed": "78.1.0"
  }
}



Model loaded; shared setup 3.6 s; platform Linux-7.0.0-31-generic-x86_64-with-glibc2.39; cpu_count 16; torch threads 4


In [4]:
# ============================================================
# FULL-CORPUS RUN AND REPRODUCTION GATE
# ============================================================
N = len(corpus)
completed = []
print(f"Running the full corpus: n={N}, replicate=1, seed=42")
result = e3.execute_run(corpus, np.arange(N), 1, 42, model)
completed.append(result)
pd.DataFrame(completed).to_csv(OUT / "runs.csv", index=False)

gate = e3.full_reference_gate(completed[0])
print("\nReproduction gate (stored outputs of notebooks 2, 3, 4 and 5 at tau = 0.40):")
print(pd.DataFrame(gate["checks"]).to_string(index=False))
validation = {"full_corpus_reference": gate, "reference_functions": reference_checks,
              "protected_artifacts_unchanged": e3.protected() == hashes,
              "completed_runs": len(completed), "tau_exact": True, "backbone_threshold_exact": True}
e3.json_write(OUT / "validation.json", validation)
assert validation["protected_artifacts_unchanged"]
if not gate["passed"]:
    metadata["status"] = "stopped_reference_mismatch"
    e3.json_write(OUT / "metadata.json", metadata)
    raise RuntimeError("STOPPED: the full-corpus CRS does not reproduce the reference; no sample is run")
print("\nREFERENCE GATE: PASSED")

Running the full corpus: n=52946, replicate=1, seed=42


n52946_r1_s42: 56635 embeddings in 121.51s


n52946_r1_s42: V=56635 E=109022 backbone=408/608 total=144.57s



Reproduction gate (stored outputs of notebooks 2, 3, 4 and 5 at tau = 0.40):
                       metric      expected        actual  absolute_tolerance  passed
                  n_documents  52946.000000  52946.000000        0.000000e+00    True
             nodes_full_graph  56635.000000  56635.000000        0.000000e+00    True
             edges_full_graph 109022.000000 109022.000000        0.000000e+00    True
      n_components_full_graph  19856.000000  19856.000000        0.000000e+00    True
                    lcc_nodes  34316.000000  34316.000000        0.000000e+00    True
                    lcc_edges 106297.000000 106297.000000        0.000000e+00    True
           lcc_fraction_nodes      0.605915      0.605915        1.000000e-10    True
           density_full_graph      0.000068      0.000068        1.000000e-10    True
               backbone_nodes    408.000000    408.000000        0.000000e+00    True
               backbone_edges    608.000000    608.000000     

In [5]:
# ============================================================
# TEN REPLICATES PER SAMPLE SIZE (SEEDS 42 TO 51)
# ============================================================
schedule = [(n, i + 1, seed) for n in e3.SIZES for i, seed in enumerate(e3.SEEDS)]
started_replicates = time.perf_counter()
for n, replicate, seed in schedule:
    positions = e3.sampled_positions(N, n, seed)
    assert len(positions) == n and len(np.unique(positions)) == n
    print(f"Running n={n}, replicate={replicate}, seed={seed}")
    result = e3.execute_run(corpus, positions, replicate, seed, model)
    completed.append(result)
    runs = pd.DataFrame(completed)
    runs.to_csv(OUT / "runs.csv", index=False)          # progressive, as the original script
    validation = {"full_corpus_reference": gate, "reference_functions": reference_checks,
                  "protected_artifacts_unchanged": e3.protected() == hashes,
                  "completed_runs": len(completed), "tau_exact": True, "backbone_threshold_exact": True}
    e3.json_write(OUT / "validation.json", validation)
    assert validation["protected_artifacts_unchanged"]
print(f"\n{len(schedule)} sample runs in {(time.perf_counter() - started_replicates) / 60:.1f} min")

Running n=500, replicate=1, seed=42


n500_r1_s42: 1604 embeddings in 3.42s


n500_r1_s42: V=1604 E=1561 backbone=0/0 total=3.58s


Running n=500, replicate=2, seed=43


n500_r2_s43: 1577 embeddings in 3.44s


...

n25000_r9_s50: V=33268 E=56962 backbone=169/236 total=88.48s


Running n=25000, replicate=10, seed=51


n25000_r10_s51: 33531 embeddings in 79.32s


n25000_r10_s51: V=33531 E=56757 backbone=172/242 total=89.67s



50 sample runs in 27.8 min


In [6]:
# ============================================================
# SAMPLE VALIDATION, SUMMARY TABLES, FIGURES AND FINAL RECORDS
# ============================================================
runs = pd.DataFrame(completed)
sample_checks = e3.validate_samples(runs, corpus, full_required=True)   # every sample re-derived from its seed
summary = e3.summarize(runs)                                             # summary.csv and runtime_summary.csv
validation = {"passed": True, "full_corpus_reference": e3.full_reference_gate(completed[0]),
              "reference_functions": reference_checks, "sample_checks": sample_checks,
              "protected_artifacts_unchanged": e3.protected() == hashes, "tau_exact": bool((runs.tau == 0.4).all()),
              "backbone_threshold_exact": bool((runs.backbone_threshold == 20).all()),
              "completed_runs": len(runs), "expected_runs": 51,
              "experiment_complete": len(runs) == 51,
              "empty_backbone_runs": int(runs.backbone_empty.sum()),
              "undefined_modularity_runs": int(runs.modularity.isna().sum())}
assert validation["protected_artifacts_unchanged"]
e3.json_write(OUT / "validation.json", validation)
metadata["status"] = "complete" if len(runs) == 51 else "reference_validated"
metadata["completed_utc"] = datetime.now(timezone.utc).isoformat()
e3.json_write(OUT / "metadata.json", metadata)
e3.figures(summary)
print(f"Status: {metadata['status']} | runs: {len(runs)} | empty backbones: {validation['empty_backbone_runs']} | "
      f"undefined modularity: {validation['undefined_modularity_runs']}")

# Table 8 layout: mean and sample SD over the ten replicates (single full-corpus run: no SD)
TABLE8_METRICS = [("nodes_full_graph", "Nodes"), ("edges_full_graph", "Edges"), ("lcc_fraction_nodes", "LCC prop."),
                  ("backbone_nodes", "Backbone nodes"), ("modularity", "Modularity"), ("runtime_total_seconds", "Time (s)")]
def table8(summary_df):
    rows = {}
    for n, group in summary_df.groupby("n_documents"):
        g = group.set_index("metric")
        rows[n] = {label: (g.loc[m, "mean"], g.loc[m, "sd"]) for m, label in TABLE8_METRICS}
    return pd.DataFrame({n: {label: (f"{m:,.3f} +/- {s:.3f}" if pd.notna(s) else f"{m:,.3f}") if pd.notna(m) else "--"
                             for label, (m, s) in r.items()} for n, r in rows.items()}).T
print("\nCRS by corpus size (this machine; mean +/- SD over ten samples, last row full corpus):")
print(table8(summary).to_string())
print("\nBackbone edges per run at n = 500 and n = 1,000:")
print(runs[runs.n_documents.isin([500, 1000])].pivot(index="replicate", columns="n_documents", values="backbone_edges").to_string())

Status: complete | runs: 51 | empty backbones: 10 | undefined modularity: 10

CRS by corpus size (this machine; mean +/- SD over ten samples, last row full corpus):
                        Nodes                   Edges        LCC prop.     Backbone nodes       Modularity          Time (s)
500      1,613.900 +/- 25.453    1,567.000 +/- 36.025  0.500 +/- 0.026    0.000 +/- 0.000               --   3.686 +/- 0.088
1000     2,866.000 +/- 44.512    3,023.900 +/- 83.777  0.518 +/- 0.017    3.200 +/- 1.033  0.291 +/- 0.250   6.621 +/- 0.109
5000    10,227.400 +/- 79.576  13,387.900 +/- 111.527  0.561 +/- 0.008   28.200 +/- 2.150  0.329 +/- 0.032  24.475 +/- 0.858
10000  17,179.200 +/- 101.543  25,092.400 +/- 131.988  0.576 +/- 0.005   74.200 +/- 2.201  0.326 +/- 0.014  42.532 +/- 1.149
25000  33,339.700 +/- 119.135  56,693.500 +/- 223.416  0.595 +/- 0.002  173.700 +/- 4.423  0.339 +/- 0.006  88.203 +/- 1.554
52946              56,635.000             109,022.000            0.606            408

## Comparison with the archived macOS run

In [7]:
# ============================================================
# STRUCTURAL QUANTITIES: THIS RUN VS THE ARCHIVED macOS RUN
# ============================================================
new_runs = pd.read_csv(OUT / "runs.csv")
old_runs = pd.read_csv(ARCHIVE / "runs.csv")
keys = ["n_documents", "replicate"]
hardware_cols = [c for c in new_runs.columns if c.startswith("runtime_") or c.endswith("_mib")]
structural_cols = [c for c in new_runs.columns if c not in hardware_cols + keys + ["sample_file"]]
merged = new_runs.merge(old_runs, on=keys, suffixes=("_new", "_macos"))
assert len(merged) == 51 == len(new_runs) == len(old_runs)

rows = []
for c in structural_cols:
    a, b = merged[f"{c}_new"], merged[f"{c}_macos"]
    if pd.api.types.is_numeric_dtype(a) and not pd.api.types.is_bool_dtype(a):
        both_nan = a.isna() & b.isna()
        diff = (a - b).abs().where(~both_nan, 0.0)
        rows.append({"column": c, "runs_compared": len(a), "max_abs_difference": float(diff.max()),
                     "runs_differing_beyond_1e-9": int((diff > 1e-9).sum()), "identical": bool((diff <= 1e-9).all())})
    else:
        rows.append({"column": c, "runs_compared": len(a), "max_abs_difference": None,
                     "runs_differing_beyond_1e-9": int((a != b).sum()), "identical": bool((a == b).all())})
structural_comparison = pd.DataFrame(rows)
print("Per-run structural quantities, 51 runs each (max |new - macOS|):")
print(structural_comparison.to_string(index=False))

differing = structural_comparison[~structural_comparison.identical]
if len(differing):
    print("\nRuns whose structural quantities differ from the archived run:")
    for c in differing.column:
        a, b = merged[f"{c}_new"], merged[f"{c}_macos"]
        bad = merged[(a != b) & ~(a.isna() & b.isna())][keys + [f"{c}_new", f"{c}_macos"]]
        print(bad.to_string(index=False))
else:
    print("\nEvery structural quantity of every run is identical to the archived macOS run "
          "(the sample files have the same SHA-256, so the samples are the same documents).")

# Summary rows (mean, sd, min, max, cv) of the structural metrics
new_summary = pd.read_csv(OUT / "summary.csv")
old_summary = pd.read_csv(ARCHIVE / "summary.csv")
s = new_summary.merge(old_summary, on=["n_documents", "metric"], suffixes=("_new", "_macos"))
s = s[~s.metric.str.startswith("runtime_") & ~s.metric.str.endswith("_mib")]
stat_diff = pd.DataFrame({stat: (s[f"{stat}_new"] - s[f"{stat}_macos"]).abs().where(~(s[f"{stat}_new"].isna() & s[f"{stat}_macos"].isna()), 0.0)
                          for stat in ["mean", "sd", "min", "max", "cv"]})
stat_diff.insert(0, "metric", s.metric.values); stat_diff.insert(0, "n_documents", s.n_documents.values)
print(f"\nsummary.csv structural rows compared: {len(s)}; largest |difference| per statistic:")
print(stat_diff[["mean", "sd", "min", "max", "cv"]].max().to_string())
print("All structural summary statistics within 1e-9 of the archived run:",
      bool((stat_diff[["mean", "sd", "min", "max", "cv"]].max() <= 1e-9).all()))

Per-run structural quantities, 51 runs each (max |new - macOS|):
                       column  runs_compared  max_abs_difference  runs_differing_beyond_1e-9  identical
             nodes_full_graph             51        0.000000e+00                           0       True
             edges_full_graph             51        0.000000e+00                           0       True
           density_full_graph             51        0.000000e+00                           0       True
      n_components_full_graph             51        0.000000e+00                           0       True
                    lcc_nodes             51        0.000000e+00                           0       True
                    lcc_edges             51        0.000000e+00                           0       True
           lcc_fraction_nodes             51        0.000000e+00                           0       True
                   modularity             51        0.000000e+00                           0       True

In [8]:
# ============================================================
# HARDWARE AND TIMINGS: MANUSCRIPT RUN (macOS) VS THIS MACHINE
# ============================================================
new_metadata = json.loads((OUT / "metadata.json").read_text())
old_metadata = json.loads((ARCHIVE / "metadata.json").read_text())
full_new = new_runs[new_runs.n_documents == N].iloc[0]
full_old = old_runs[old_runs.n_documents == N].iloc[0]

side = pd.DataFrame({
    "manuscript run (macOS)": {
        "platform": old_metadata["platform"], "cpu_count": old_metadata["cpu_count"], "python": old_metadata["python"],
        "torch": old_metadata["packages"]["torch"], "sentence-transformers": old_metadata["packages"]["sentence-transformers"],
        "numpy": old_metadata["packages"]["numpy"], "torch threads": 4,
        "full corpus: total runtime (s)": round(full_old.runtime_total_seconds, 1),
        "full corpus: embedding 56,635 terms (s)": round(full_old.runtime_embedding_seconds, 1),
        "full corpus: graph construction (s)": round(full_old.runtime_graph_construction_seconds, 1),
        "full corpus: metrics (s)": round(full_old.runtime_metrics_seconds, 1),
        "full corpus: peak RSS (MiB)": round(full_old.peak_memory_rss_mib, 1),
        "started_utc": old_metadata["started_utc"]},
    "this run": {
        "platform": new_metadata["platform"], "cpu_count": new_metadata["cpu_count"], "python": new_metadata["python"],
        "torch": new_metadata["packages"]["torch"], "sentence-transformers": new_metadata["packages"]["sentence-transformers"],
        "numpy": new_metadata["packages"]["numpy"], "torch threads": 4,
        "full corpus: total runtime (s)": round(full_new.runtime_total_seconds, 1),
        "full corpus: embedding 56,635 terms (s)": round(full_new.runtime_embedding_seconds, 1),
        "full corpus: graph construction (s)": round(full_new.runtime_graph_construction_seconds, 1),
        "full corpus: metrics (s)": round(full_new.runtime_metrics_seconds, 1),
        "full corpus: peak RSS (MiB)": round(full_new.peak_memory_rss_mib, 1),
        "started_utc": new_metadata["started_utc"]}})
print(side.to_string())

t = new_summary[new_summary.metric == "runtime_total_seconds"].merge(
    old_summary[old_summary.metric == "runtime_total_seconds"], on="n_documents", suffixes=("_new", "_macos"))
timing = pd.DataFrame({"n_documents": t.n_documents,
                       "macOS mean (s)": t.mean_macos.round(1), "macOS sd": t.sd_macos.round(1),
                       "this machine mean (s)": t.mean_new.round(1), "this machine sd": t.sd_new.round(1),
                       "ratio this/macOS": (t.mean_new / t.mean_macos).round(2)})
print("\nTotal runtime per size (mean and sample SD over ten samples; full corpus: one run):")
print(timing.to_string(index=False))
print("\nRuntimes and peak memory are hardware dependent: the manuscript (Table 8, Section IV-D2 and Appendix A-D) "
      "reports the macOS values in the left column; the structural quantities do not depend on the machine.")

                                                      manuscript run (macOS)                                      this run
platform                                 macOS-26.5.1-arm64-arm-64bit-Mach-O  Linux-7.0.0-31-generic-x86_64-with-glibc2.39
cpu_count                                                                  8                                            16
python                                                                3.13.5                                        3.12.3
torch                                                                  2.8.0                                     2.8.0+cpu
sentence-transformers                                                  5.1.1                                         5.1.1
numpy                                                                  2.1.3                                         2.1.3
torch threads                                                              4                                             4
full corpus: tot

## Check against the manuscript

In [9]:
# ============================================================
# CHECK AGAINST THE MANUSCRIPT (values transcribed from main.tex)
# ============================================================
def fmt_like(value, template):
    """Format value with the precision (decimals, thousands separator) of the manuscript string."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "--"
    if template in (None, "--"):
        return f"{value:.3f}"
    decimals = len(template.split(".")[1]) if "." in template else 0
    return f"{value:,.{decimals}f}" if "," in template else f"{value:.{decimals}f}"

def stat(df, n, metric, which):
    row = df[(df.n_documents == n) & (df.metric == metric)].iloc[0]
    return row[which]

# Table 8 of the manuscript (Section IV-D2), per size: mean +/- sd
TABLE8 = {  # n: nodes, edges, LCC prop., backbone nodes, modularity, time; each as (mean, sd) strings
    500:   [("1,614", "25"), ("1,567", "36"), ("0.500", "0.026"), ("0", "0"), ("--", None), ("2.5", "0.2")],
    1000:  [("2,866", "45"), ("3,024", "84"), ("0.518", "0.017"), ("3.2", "1.0"), ("0.291", "0.250"), ("4.9", "0.5")],
    5000:  [("10,227", "80"), ("13,388", "112"), ("0.561", "0.008"), ("28.2", "2.1"), ("0.329", "0.032"), ("15.5", "1.3")],
    10000: [("17,179", "102"), ("25,092", "132"), ("0.576", "0.005"), ("74.2", "2.2"), ("0.326", "0.014"), ("26.4", "0.7")],
    25000: [("33,340", "119"), ("56,694", "223"), ("0.595", "0.002"), ("173.7", "4.4"), ("0.339", "0.006"), ("53.8", "2.6")],
    52946: [("56,635", None), ("109,022", None), ("0.606", None), ("408", None), ("0.356", None), ("92.4", None)],
}
check = []
for n, entries in TABLE8.items():
    for (metric, label), (m_str, sd_str) in zip(TABLE8_METRICS, entries):
        hardware = metric.startswith("runtime_")
        new_m, new_sd = stat(new_summary, n, metric, "mean"), stat(new_summary, n, metric, "sd")
        old_m, old_sd = stat(old_summary, n, metric, "mean"), stat(old_summary, n, metric, "sd")
        manuscript = m_str if sd_str is None else f"{m_str} +/- {sd_str}"
        computed = fmt_like(new_m, m_str) if sd_str is None else f"{fmt_like(new_m, m_str)} +/- {fmt_like(new_sd, sd_str)}"
        archived = fmt_like(old_m, m_str) if sd_str is None else f"{fmt_like(old_m, m_str)} +/- {fmt_like(old_sd, sd_str)}"
        flag = "match" if computed == manuscript else "differs"
        note = ""
        if hardware:
            note = f"hardware dependent; archived macOS run: {archived} ({'match' if archived == manuscript else 'differs'})"
        check.append({"location": f"Table 8, n = {n:,}", "quantity": label, "manuscript": manuscript,
                      "computed": computed, "flag": flag, "note": note})

# Section IV-D2 text and gate values
full = new_runs[new_runs.n_documents == N].iloc[0]
full_macos = old_runs[old_runs.n_documents == N].iloc[0]
def add(location, quantity, manuscript, value, template, hardware_value=None):
    computed = fmt_like(value, template)
    row = {"location": location, "quantity": quantity, "manuscript": manuscript, "computed": computed,
           "flag": "match" if computed == manuscript else "differs", "note": ""}
    if hardware_value is not None:
        arch = fmt_like(hardware_value, template)
        row["note"] = f"hardware dependent; archived macOS run: {arch} ({'match' if arch == manuscript else 'differs'})"
    check.append(row)

add("Sec. IV-D2 (text)", "analysed documents", "52,946", float(N), "52,946")
add("Sec. IV-D2 (gate)", "full-corpus nodes", "56,635", float(full.nodes_full_graph), "56,635")
add("Sec. IV-D2 (gate)", "full-corpus edges", "109,022", float(full.edges_full_graph), "109,022")
add("Sec. IV-D2 (gate)", "backbone nodes", "408", float(full.backbone_nodes), "408")
add("Sec. IV-D2 (gate)", "backbone edges", "608", float(full.backbone_edges), "608")
add("Sec. IV-D2 (gate)", "backbone modularity", "0.356044", float(full.modularity), "0.356044")
add("Sec. IV-D2 (gate)", "backbone communities", "7", float(full.n_communities), "7")
add("Sec. IV-D2 (text)", "LCC fraction at 500 documents (sd)", "0.500 (0.026)",
    None, None); check[-1].update(computed=f"{fmt_like(stat(new_summary, 500, 'lcc_fraction_nodes', 'mean'), '0.500')} ({fmt_like(stat(new_summary, 500, 'lcc_fraction_nodes', 'sd'), '0.026')})")
check[-1]["flag"] = "match" if check[-1]["computed"] == check[-1]["manuscript"] else "differs"
add("Sec. IV-D2 (text)", "LCC fraction at 25,000 documents (sd)", "0.595 (0.002)", None, None)
check[-1].update(computed=f"{fmt_like(stat(new_summary, 25000, 'lcc_fraction_nodes', 'mean'), '0.595')} ({fmt_like(stat(new_summary, 25000, 'lcc_fraction_nodes', 'sd'), '0.002')})")
check[-1]["flag"] = "match" if check[-1]["computed"] == check[-1]["manuscript"] else "differs"
add("Sec. IV-D2 (text)", "LCC fraction, full corpus", "0.606", float(full.lcc_fraction_nodes), "0.606")
add("Sec. IV-D2 (text)", "modularity at 5,000 documents (sd)", "0.329 (0.032)", None, None)
check[-1].update(computed=f"{fmt_like(stat(new_summary, 5000, 'modularity', 'mean'), '0.329')} ({fmt_like(stat(new_summary, 5000, 'modularity', 'sd'), '0.032')})")
check[-1]["flag"] = "match" if check[-1]["computed"] == check[-1]["manuscript"] else "differs"
add("Sec. IV-D2 (text)", "modularity at 25,000 documents (sd)", "0.339 (0.006)", None, None)
check[-1].update(computed=f"{fmt_like(stat(new_summary, 25000, 'modularity', 'mean'), '0.339')} ({fmt_like(stat(new_summary, 25000, 'modularity', 'sd'), '0.006')})")
check[-1]["flag"] = "match" if check[-1]["computed"] == check[-1]["manuscript"] else "differs"
add("Sec. IV-D2 (text)", "runtime range, smallest size (s)", "2.5", stat(new_summary, 500, "runtime_total_seconds", "mean"), "2.5",
    hardware_value=stat(old_summary, 500, "runtime_total_seconds", "mean"))
add("Sec. IV-D2 (text) and App. A-D", "full run total runtime (s)", "92", full.runtime_total_seconds, "92",
    hardware_value=full_macos.runtime_total_seconds)
add("Sec. IV-D2 (text) and App. A-D", "full run embedding time (s)", "75", full.runtime_embedding_seconds, "75",
    hardware_value=full_macos.runtime_embedding_seconds)
add("Sec. IV-D2 (text)", "full run graph construction (s)", "9.5", full.runtime_graph_construction_seconds, "9.5",
    hardware_value=full_macos.runtime_graph_construction_seconds)
add("Sec. IV-D2 (text) and App. A-D", "vocabulary embedded in the full run (terms)", "56,635", float(full.vocabulary_size), "56,635")
add("App. A-D", "full run peak memory (MiB)", "859", full.peak_memory_rss_mib, "859", hardware_value=full_macos.peak_memory_rss_mib)
add("Sec. IV-D2 and Sec. III-D", "500-document samples with at least one backbone edge", "0",
    float((new_runs[new_runs.n_documents == 500].backbone_edges > 0).sum()), "0")
add("Sec. IV-D2 (text)", "mean backbone nodes at 1,000 documents", "3.2", stat(new_summary, 1000, "backbone_nodes", "mean"), "3.2")
for n, ms in [(5000, "28"), (10000, "74"), (25000, "174")]:
    add("Sec. IV-D2 (text)", f"backbone nodes at {n:,} documents (rounded mean)", ms, stat(new_summary, n, "backbone_nodes", "mean"), ms)
n1000 = new_runs[new_runs.n_documents == 1000]
one_or_two = int(n1000.backbone_edges.isin([1, 2]).sum())
check.append({"location": "Table 8 footnote", "quantity": "1,000-document samples with a backbone of one or two edges",
              "manuscript": "several", "computed": f"{one_or_two} of 10", "flag": "match" if one_or_two >= 2 else "differs", "note": ""})
check.append({"location": "App. A-D and Table 8 caption", "quantity": "hardware of the reported run",
              "manuscript": "8-core Apple silicon (arm64) laptop, macOS, CPU only",
              "computed": f"{new_metadata['platform']}, {new_metadata['cpu_count']} cores, CPU only",
              "flag": "differs", "note": f"hardware dependent; archived run: {old_metadata['platform']}, {old_metadata['cpu_count']} cores (match)"})

manuscript_check = pd.DataFrame(check)
pd.set_option("display.max_colwidth", 90)
print(manuscript_check.to_string(index=False))
n_match = int((manuscript_check.flag == "match").sum())
n_hw = int(manuscript_check.note.str.startswith("hardware").sum())
n_diff_hw = int(((manuscript_check.flag == "differs") & manuscript_check.note.str.startswith("hardware")).sum())
print(f"\n{n_match} of {len(manuscript_check)} rows match. Differing rows: {len(manuscript_check) - n_match} "
      f"(hardware-dependent timing, memory or platform rows: {n_diff_hw}; other: {len(manuscript_check) - n_match - n_diff_hw}). "
      f"The table has {n_hw} hardware-dependent rows; for each of them the note states whether the archived macOS run reproduces the manuscript value.")

                      location                                                   quantity                                           manuscript                                                         computed    flag                                                                                   note
              Table 8, n = 500                                                      Nodes                                         1,614 +/- 25                                                     1,614 +/- 25   match                                                                                       
              Table 8, n = 500                                                      Edges                                         1,567 +/- 36                                                     1,567 +/- 36   match                                                                                       
              Table 8, n = 500                                                  LCC prop.  